# 09 — Aggregator — federated AU

| | |
|---|---|
| **runs at** | the aggregator |
| **reads** | both `_stats.npz`, both `_counts.csv` |
| **writes** | `federated_pvclust_edges.csv` |

Sum the counts, fit the multiscale curve once, centrally.

**What AU is, in one paragraph.** An ordinary bootstrap probability is biased: it
systematically overstates or understates how often a cluster is the true one,
because the region boundary in parameter space is curved. Shimodaira's multiscale
bootstrap resamples at several *sample sizes*, fits how the probability changes
with scale, and extrapolates to a point no bootstrap can sample — a negative
variance — where the bias cancels. That extrapolated value is the approximately
unbiased *p*-value.

This is why several resampling sizes exist at all, and why the `n` column in the
counts matters.

In [ ]:
import os, subprocess
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

# Every notebook reads and writes here, so the chain hands work along. In a real
# deployment this directory would be on a different machine for each cohort, and
# only the files named 'what this writes' would ever cross the boundary.
WORK = Path('sle-run'); WORK.mkdir(exist_ok=True); os.chdir(WORK)

def run(cmd):
    """Run one pvclust-py command and echo it, so the notebook shows the command
    line rather than hiding it behind a function."""
    print('$ ' + ' '.join(cmd))
    r = subprocess.run(cmd, capture_output=True, text=True)
    print((r.stdout + r.stderr).strip()[-1800:])
    if r.returncode:
        raise SystemExit(f'command failed: {cmd}')

def show(png):
    """Render a written figure inline. matplotlib only, so this works in CI too."""
    if not Path(png).exists():
        print(f'(missing {png})'); return
    fig, ax = plt.subplots(figsize=(13, 13))
    ax.imshow(plt.imread(png)); ax.axis('off'); ax.set_title(png, fontsize=9)
    plt.show()

In [ ]:
run(['pvclust-py', 'aggregate-trees',
     '--labels', 'cohort-A_labels.txt',
     '--stats', 'cohort-A_stats.npz',
     '--stats', 'cohort-B_stats.npz',
     '--counts', 'cohort-A_counts.csv',
     '--counts', 'cohort-B_counts.csv',
     '--dist', 'minkowski', '--linkage', 'ward.D2', '--alpha', '0.95'])

## The federated result

`n_projects` says how many cohorts measured each cluster. A cluster measured by
fewer than all of them is weaker evidence than its AU suggests, and the command
warns when that happens.

Again: **check `df` before trusting `au`**.

In [ ]:
fed = pd.read_csv('federated_pvclust_edges.csv').sort_values('au', ascending=False)
cols = [c for c in ['n_members', 'au', 'bp', 'se_au', 'df', 'n_projects'] if c in fed]
print(f'{len(fed)} clusters, {(fed.au >= 0.95).sum()} at AU >= 0.95, '
      f'{(fed.df == 0).sum()} with a degenerate fit')
print()
print(fed[cols].head(10).to_string(index=False))

---
**Next:** `10` and `11`. Each cohort applies this back to its own data and asks
whether joining was worth it.